In [80]:
## importing library
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [81]:
## loading data
match=pd.read_csv("matches.csv")
delivery=pd.read_csv("deliveries.csv")

ipl=delivery.merge(match,left_on='match_id',right_on='id')
ipl.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN


In [82]:
## scatter plot : find the Top 50 batsmen by runs and then plot their Average vs Strike Rate

# Top 50 batsmen by runs
top50 = (
    ipl.groupby('batsman')['batsman_runs']
    .sum()
    .sort_values(ascending=False)
    .head(50)
    .index
)

# Filter dataset
new_ipl = ipl[ipl['batsman'].isin(top50)]

# Runs
runs = new_ipl.groupby('batsman')['batsman_runs'].sum()

# Balls faced
balls = new_ipl.groupby('batsman')['batsman_runs'].count()

# Strike Rate
sr = (runs / balls) * 100

# Outs
outs = (
    new_ipl[new_ipl['player_dismissed'].notna()]
    .groupby('player_dismissed')
    .size()
)

# Average
avg = runs / outs

# Convert to DataFrame
avg = avg.reset_index()
avg.columns = ['batsman', 'avg']

sr = sr.reset_index()
sr.columns = ['batsman', 'sr']

# Merge
final = avg.merge(sr, on='batsman')

# Add runs for bubble size
runs = runs.reset_index()
runs.columns = ['batsman', 'runs']

final = final.merge(runs, on='batsman')

In [83]:
## plotting scatter plot
trace = go.Scatter(
    x=final['avg'],
    y=final['sr'],
    mode='markers+text',
    text=final['batsman'],
    textposition='top center',
    marker={'color': final['sr'],'colorscale': 'Viridis','showscale': True}
)

layout = go.Layout(
    title='Top 50 IPL Batsmen: Avg vs SR',
    xaxis_title='Average',
    yaxis_title='Strike Rate'
)
data=[trace]
fig = go.Figure(data=data, layout=layout)

fig.show()

In [84]:
## Line Plot : Year by year batsman performance

single=ipl[ipl['batsman']=="V Kohli"]
performance=single.groupby('season')['batsman_runs'].sum().reset_index()
performance

,season,batsman_runs
0,2008,165
1,2009,246
2,2010,307
3,2011,557
4,2012,364
5,2013,639
6,2014,359
7,2015,505
8,2016,973
9,2017,308


In [85]:
## plotting
trace = go.Scatter(
    x=performance['season'],
    y=performance['batsman_runs'],
    mode='lines+markers',
    name='V Kohli'
)

layout = go.Layout(
    title='V Kohli Season-wise Performance',
    xaxis_title='Season',
    yaxis_title='Runs'
)
data=[trace]
fig = go.Figure(data=data, layout=layout)
fig.show()

In [86]:
## multiple line chart

vk = ipl[ipl['batsman'] == "V Kohli"]
vk_perf = vk.groupby('season')['batsman_runs'].sum().reset_index()

rs = ipl[ipl['batsman'] == "RG Sharma"]
rs_perf = rs.groupby('season')['batsman_runs'].sum().reset_index()

trace1 = go.Scatter(
    x=vk_perf['season'],
    y=vk_perf['batsman_runs'],
    mode='lines+markers',
    name='Virat Kohli'
)

trace2 = go.Scatter(
    x=rs_perf['season'],
    y=rs_perf['batsman_runs'],
    mode='lines+markers',
    name='Rohit Sharma'
)

layout = go.Layout(
    title='Season-wise Performance Comparison',
    xaxis_title='Season',
    yaxis_title='Runs'
)
data=[trace1, trace2]
fig = go.Figure(data=data, layout=layout)
fig.show()

In [87]:
## funtion to compare run

def plot_batsmen_performance(ipl, *batsmen):
    fig = go.Figure()

    for batsman in batsmen:
        data = ipl[ipl['batsman'] == batsman]
        performance = data.groupby('season')['batsman_runs'].sum().reset_index()

        fig.add_trace(go.Scatter(
            x=performance['season'],
            y=performance['batsman_runs'],
            mode='lines+markers',
            name=batsman
        ))

    fig.update_layout(
        title='Season-wise Batsmen Performance Comparison',
        xaxis_title='Season',
        yaxis_title='Runs'
    )

    fig.show()

plot_batsmen_performance(ipl, "V Kohli", "RG Sharma", "DA Warner","AB de Villiers")


In [88]:
## Bar Plot

top10 = ipl.groupby('batsman')['batsman_runs'].sum().reset_index().sort_values(by='batsman_runs', ascending=False).head(10)

score = ipl.groupby('batsman')['batsman_runs'].sum().reset_index().sort_values(by='batsman_runs', ascending=False).head(10)

trace=go.Bar(x=score['batsman'],y=score['batsman_runs'],marker={'color': score['batsman_runs']})
layout=go.Layout(title='Top 10 ipl batsman',xaxis_title='Batsman',yaxis_title='Runs')

data=[trace]
fig=go.Figure(data=data,layout=layout)

fig.show()


In [89]:
## Stacked Bar Chart : Runs comparison of 2–3 batsmen stacked by season.

vk = ipl[ipl['batsman'] == "V Kohli"].groupby('season')['batsman_runs'].sum().reset_index()
rs = ipl[ipl['batsman'] == "RG Sharma"].groupby('season')['batsman_runs'].sum().reset_index()

trace1 = go.Bar(
    x=vk['season'],
    y=vk['batsman_runs'],
    name='V Kohli'
)

trace2 = go.Bar(
    x=rs['season'],
    y=rs['batsman_runs'],
    name='Rohit Sharma'
)

layout = go.Layout(
    title='Stacked Runs Comparison',
    xaxis_title='Season',
    yaxis_title='Runs',
    barmode='stack'
)

fig = go.Figure(data=[trace1, trace2], layout=layout)
fig.show()

In [90]:
## OVERLAID (GROUPED / SIDE-BY-SIDE)
layout = go.Layout(
    title='Overlayed Runs Comparison',
    xaxis_title='Season',
    yaxis_title='Runs',
    barmode='group'
)
fig = go.Figure(data=[trace1, trace2], layout=layout)
fig.show()

In [91]:
## TRUE OVERLAY (transparent bars) If you want bars on top of each other

trace1 = go.Bar(
    x=vk['season'],
    y=vk['batsman_runs'],
    name='V Kohli',
    opacity=0.7
)

trace2 = go.Bar(
    x=rs['season'],
    y=rs['batsman_runs'],
    name='Rohit Sharma',
    opacity=0.5
)

layout = go.Layout(
    title='Overlayed Bars (Transparent)',
    barmode='overlay',
    xaxis_title='Season',
    yaxis_title='Runs'
)

fig = go.Figure(data=[trace1, trace2], layout=layout)
fig.show()



| Mode                 | How bars look                        | Use case                                              | Key behavior                              |
| -------------------- | ------------------------------------ | ----------------------------------------------------- | ----------------------------------------- |
| `stack`              | Bars are placed on top of each other | Total contribution (e.g., runs per player per season) | Values are added vertically               |
| `group`              | Bars are side-by-side                | Comparison between categories                         | No overlap, clean comparison              |
| `overlay`            | Bars overlap each other              | Direct comparison in same space                       | Requires opacity to see both              |
| `relative` (default) | Like stacked bars                    | General use, especially with positive/negative data   | Handles stacking with direction (pos/neg) | **bold text**


In [92]:
## Bubble Plot

trace = go.Scatter(
    x=final['avg'],
    y=final['sr'],
    mode='markers',
    text=final['batsman'],
    marker=dict(
        size=final['runs'] / 50,
        color=final['runs'],
        showscale=True
    )
)

layout = go.Layout(
    title='IPL Bubble Plot: Avg vs SR',
    xaxis_title='Average',
    yaxis_title='Strike Rate'
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

In [93]:
sixes = ipl[ipl['batsman_runs'] == 6].groupby('batsman').size().reset_index(name='sixes')

final = final.drop(columns=['sixes'], errors='ignore')
final = final.merge(sixes, on='batsman', how='left')
final['sixes'] = final['sixes'].fillna(0)

final_50 = final.sort_values(by='sixes', ascending=False).head(50)

trace = go.Scatter(
    x=final_50['avg'],
    y=final_50['sr'],
    mode='markers',
    text=final_50['batsman'],
    marker=dict(
        size=final_50['sixes'] ,
        color=final_50['sixes'],
        colorscale='Viridis',
        showscale=True
    )
)

layout = go.Layout(
    title='IPL Bubble Plot: Avg vs SR (Top 50 Six Hitters)',
    xaxis_title='Average',
    yaxis_title='Strike Rate'
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

In [94]:
## boxplot Innings-wise Total Runs (IPL)

innings_runs = ipl.groupby(['match_id', 'inning'])['total_runs'].sum().reset_index()

trace = go.Box(x=innings_runs['inning'],y=innings_runs['total_runs'])

layout = go.Layout(
    title='IPL Innings-wise Total Runs Distribution',
    xaxis_title='Innings',
    yaxis_title='Total Runs per Match'
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

In [95]:
season_innings = ipl.groupby(['season', 'inning'])['total_runs'].sum().reset_index()

trace = go.Box(
    x=season_innings['season'],
    y=season_innings['total_runs']
)

layout = go.Layout(
    title='Season-wise Innings Total Runs Distribution',
    xaxis_title='Season',
    yaxis_title='Total Runs'
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

In [96]:
ipl_2017 = ipl[ipl['season'] == 2017]

innings_2017 = ipl_2017.groupby(['match_id', 'inning'])['total_runs'].sum().reset_index()

trace = go.Box(
    y=innings_2017['total_runs']
)

layout = go.Layout(
    title='IPL 2017: Innings Total Runs Distribution',
    yaxis_title='Total Runs per Innings'
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

In [121]:
## Histogram

x=delivery.groupby('batsman')['batsman_runs'].count()>150
x=x[x].index.tolist()

new=delivery[delivery['batsman'].isin(x)]

runs=new.groupby('batsman')['batsman_runs'].sum()
balls=new.groupby('batsman')['batsman_runs'].count()

sr=(runs/balls) * 100
sr=sr.reset_index()

trace=go.Histogram(x=sr['batsman_runs'],xbins={'size':5,'start':20,'end':150})
data=[trace]
layout=go.Layout(title='Strike rate Analysis',xaxis_title='Strike rate')

fig=go.Figure(data=data,layout=layout)
fig.show()


In [98]:
trace1 = go.Histogram(x=ipl[ipl['batsman_runs'] == 4]['batsman_runs'], name='Fours')
trace2 = go.Histogram(x=ipl[ipl['batsman_runs'] == 6]['batsman_runs'], name='Sixes')

layout = go.Layout(
    barmode='overlay',
    title='Boundaries Distribution'
)

fig = go.Figure(data=[trace1, trace2], layout=layout)
fig.show()

In [110]:
## distplot
import numpy as np
import plotly.figure_factory as ff

clean_avg = final['avg'].replace([np.inf, -np.inf], np.nan).dropna()

hist_data = [clean_avg,final['sr']]
group_labels = ['Average','Strike rate']

fig = ff.create_distplot(hist_data, group_labels)

fig.show()

In [129]:
## Heatmap

sixes = ipl[ipl['batsman_runs'] == 6]

six = sixes.groupby(['batting_team', 'over']).count().reset_index()

trace=go.Heatmap(x=six['batting_team'],y=six['over'],z=six['batsman_runs'])
data=[trace]

layout=go.Layout(title='Six Heatmap')

fig=go.Figure(data=data,layout=layout)
fig.show()

In [122]:
## Heatmaps
heat = ipl.pivot_table(
    index='batsman',
    columns='season',
    values='batsman_runs',
    aggfunc='sum'
).fillna(0)

trace = go.Heatmap(
    z=heat.values,
    x=heat.columns,
    y=heat.index
)

layout = go.Layout(
    title='Batsman vs Season Heatmap (Runs)',
    xaxis_title='Season',
    yaxis_title='Batsman'
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

In [123]:
team_heat = ipl.pivot_table(
    index='batting_team',
    columns='season',
    values='total_runs',
    aggfunc='sum'
).fillna(0)

trace = go.Heatmap(
    z=team_heat.values,
    x=team_heat.columns,
    y=team_heat.index
)

layout = go.Layout(
    title='Team vs Season Runs Heatmap'
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

In [124]:
opp_heat = ipl.pivot_table(
    index='batsman',
    columns='bowling_team',
    values='batsman_runs',
    aggfunc='sum'
).fillna(0)

trace = go.Heatmap(
    z=opp_heat.values,
    x=opp_heat.columns,
    y=opp_heat.index
)

layout = go.Layout(
    title='Batsman vs Bowler Team Heatmap'
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()